In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization,
    GlobalAveragePooling2D
)
from tensorflow.keras.applications import (
    MobileNetV2,
    EfficientNetB0,
    ResNet50,
    DenseNet121
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

In [3]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [4]:
encoder = LabelEncoder()
train_df["label"] = encoder.fit_transform(train_df["disease"])
valid_df["label"] = encoder.transform(valid_df["disease"])
test_df["label"] = encoder.transform(test_df["disease"])

In [5]:
print(train_df["label"].unique())
print(valid_df["label"].unique())
print(test_df["label"].unique())

[0 1 4 5 2 6 3]
[3 4 5 2 1 0 6]
[1 4 2 6 5 0 3]


In [6]:
print(train_df.groupby("label").size())
print(valid_df.groupby("label").size())
print(test_df.groupby("label").size())

label
0     229
1     360
2     769
3      81
4    4693
5     779
6      99
dtype: int64
label
0      49
1      77
2     165
3      17
4    1006
5     167
6      21
dtype: int64
label
0      49
1      77
2     165
3      17
4    1006
5     167
6      22
dtype: int64


In [7]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 32

In [8]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [9]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

In [10]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [11]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [12]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [13]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

print(weights)

class_weights = dict(zip(np.unique(train_df["label"]), weights))
print(class_weights)

[ 4.37305053  2.78174603  1.30224782 12.3633157   0.21338772  1.2855309
 10.11544012]
{np.int64(0): np.float64(4.37305053025577), np.int64(1): np.float64(2.7817460317460316), np.int64(2): np.float64(1.3022478172023035), np.int64(3): np.float64(12.36331569664903), np.int64(4): np.float64(0.21338772031292808), np.int64(5): np.float64(1.285530900421786), np.int64(6): np.float64(10.115440115440116)}


In [14]:
for images, labels in train_ds.take(1):
    print(
        tf.reduce_min(images).numpy()
    )
    print(
        tf.reduce_max(images).numpy()
    )

0.0
1.0


In [15]:
basic_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D((2,2)),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

C:\Users\acer\miniconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
basic_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [37]:
history_basic = basic_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 351ms/step - accuracy: 0.4385 - loss: 1.9649 - val_accuracy: 0.3828 - val_loss: 1.6819
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 350ms/step - accuracy: 0.3425 - loss: 1.7732 - val_accuracy: 0.2850 - val_loss: 1.8233
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 351ms/step - accuracy: 0.3823 - loss: 1.6394 - val_accuracy: 0.4794 - val_loss: 1.3780
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 355ms/step - accuracy: 0.3421 - loss: 1.6073 - val_accuracy: 0.1924 - val_loss: 2.0200
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 349ms/step - accuracy: 0.3545 - loss: 1.5695 - val_accuracy: 0.3715 - val_loss: 1.5333
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 79s 350ms/step - accuracy: 0.3977 - loss: 1.5033 - val_accuracy: 0.3555 - val_loss: 1.5977
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 357ms/step - accuracy: 0.4260 - loss: 1.4712 - val_accuracy: 0.4261 - val_loss: 1.4017
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 80s 356ms/step - accuracy: 0.4536 - loss: 1.4180 - val

In [38]:
test_loss, test_acc = basic_cnn.evaluate(test_ds)
print("Accuracy:",test_acc)
print("Loss:",test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 112ms/step - accuracy: 0.5010 - loss: 1.2069
Accuracy: 0.5009980201721191
Loss: 1.206936240196228


In [17]:
deep_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        256,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [18]:
deep_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [41]:
history_deep = deep_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 118s 525ms/step - accuracy: 0.3907 - loss: 1.8590 - val_accuracy: 0.2130 - val_loss: 1.6873
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 118s 527ms/step - accuracy: 0.2673 - loss: 1.8646 - val_accuracy: 0.3628 - val_loss: 1.5512
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 118s 529ms/step - accuracy: 0.3981 - loss: 1.7296 - val_accuracy: 0.3961 - val_loss: 1.4885
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 116s 517ms/step - accuracy: 0.3857 - loss: 1.6388 - val_accuracy: 0.3768 - val_loss: 1.5371
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 115s 516ms/step - accuracy: 0.4665 - loss: 1.5119 - val_accuracy: 0.4594 - val_loss: 1.3845
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 115s 516ms/step - accuracy: 0.4287 - loss: 1.4720 - val_accuracy: 0.5905 - val_loss: 1.1959
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 117s 525ms/step - accuracy: 0.4679 - loss: 1.4596 - val_accuracy: 0.2949 - val_loss: 1.8460
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 119s 534ms/step - accuracy: 0.4866 -

In [42]:
test_loss, test_acc = deep_cnn.evaluate(test_ds)
print("Accuracy:",test_acc)
print("Loss:",test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 167ms/step - accuracy: 0.4597 - loss: 1.3373
Accuracy: 0.45974716544151306
Loss: 1.3373124599456787


In [19]:
cnn_bn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    BatchNormalization(),
    MaxPooling2D(2,2),
    GlobalAveragePooling2D(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [20]:
cnn_bn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [23]:
history_bn = cnn_bn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 148s 660ms/step - accuracy: 0.4083 - loss: 1.7067 - val_accuracy: 0.0333 - val_loss: 3.3734
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 146s 655ms/step - accuracy: 0.4552 - loss: 1.4899 - val_accuracy: 0.0779 - val_loss: 2.7278
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 146s 657ms/step - accuracy: 0.4971 - loss: 1.4050 - val_accuracy: 0.4334 - val_loss: 1.5971
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 146s 655ms/step - accuracy: 0.4910 - loss: 1.4014 - val_accuracy: 0.1511 - val_loss: 2.3899
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 148s 665ms/step - accuracy: 0.5140 - loss: 1.3540 - val_accuracy: 0.3662 - val_loss: 1.6620
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 150s 673ms/step - accuracy: 0.5204 - loss: 1.3018 - val_accuracy: 0.5060 - val_loss: 1.3970
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 148s 665ms/step - accuracy: 0.5231 - loss: 1.2862 - val_accuracy: 0.3901 - val_loss: 1.8580
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 149s 667ms/step - accuracy: 0.4997 -

In [24]:
test_loss, test_acc = cnn_bn.evaluate(test_ds)
print("Accuracy:",test_acc)
print("Loss:",test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 161ms/step - accuracy: 0.4950 - loss: 1.3026
Accuracy: 0.49500998854637146
Loss: 1.3025752305984497


In [21]:
cnn_dropout = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dropout(0.5),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [22]:
cnn_dropout.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [49]:
history_dropout = cnn_dropout.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 464ms/step - accuracy: 0.3064 - loss: 1.9749 - val_accuracy: 0.1338 - val_loss: 1.9598
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 108s 481ms/step - accuracy: 0.3845 - loss: 1.8392 - val_accuracy: 0.0832 - val_loss: 2.0314
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 464ms/step - accuracy: 0.1444 - loss: 1.9192 - val_accuracy: 0.1844 - val_loss: 1.8989
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 103s 459ms/step - accuracy: 0.3481 - loss: 1.8183 - val_accuracy: 0.5093 - val_loss: 1.3498
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 106s 471ms/step - accuracy: 0.3020 - loss: 1.8136 - val_accuracy: 0.3875 - val_loss: 1.4839
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 104s 462ms/step - accuracy: 0.3572 - loss: 1.7639 - val_accuracy: 0.1425 - val_loss: 1.6883
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 103s 458ms/step - accuracy: 0.3782 - loss: 1.7517 - val_accuracy: 0.3276 - val_loss: 1.6498
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 102s 456ms/step - accuracy: 0.4138 -

In [50]:
test_loss, test_acc = cnn_dropout.evaluate(test_ds)
print("Accuracy:",test_acc)
print("Loss:",test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 7s 144ms/step - accuracy: 0.4311 - loss: 1.3529
Accuracy: 0.43113771080970764
Loss: 1.3528671264648438


In [23]:
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

In [24]:
mobilenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(
        128,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [25]:
mobilenet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [17]:
history_mobile = mobilenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 51s 217ms/step - accuracy: 0.4455 - loss: 1.5792 - val_accuracy: 0.6258 - val_loss: 1.0951
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 49s 216ms/step - accuracy: 0.5472 - loss: 1.2755 - val_accuracy: 0.5679 - val_loss: 1.1942
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 49s 216ms/step - accuracy: 0.5700 - loss: 1.1633 - val_accuracy: 0.6158 - val_loss: 1.0813
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 50s 220ms/step - accuracy: 0.5879 - loss: 1.0934 - val_accuracy: 0.5832 - val_loss: 1.1493
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 49s 216ms/step - accuracy: 0.6024 - loss: 1.0072 - val_accuracy: 0.6858 - val_loss: 0.9024
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 49s 218ms/step - accuracy: 0.6054 - loss: 1.0025 - val_accuracy: 0.6897 - val_loss: 0.8683
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 49s 218ms/step - accuracy: 0.6324 - loss: 0.9484 - val_accuracy: 0.5080 - val_loss: 1.3587
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 51s 224ms/step - accuracy: 0.6183 - loss: 0.9599 - val

In [18]:
test_loss, test_acc = mobilenet.evaluate(test_ds)
print("Accuracy:",test_acc)
print("Loss:",test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 180ms/step - accuracy: 0.5749 - loss: 1.1823
Accuracy: 0.57485032081604
Loss: 1.182260513305664


In [26]:
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = data_augmentation(image)
    return image, label

In [27]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [28]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [29]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [30]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

print(weights)

class_weights = dict(zip(np.unique(train_df["label"]), weights))
print(class_weights)

[ 4.37305053  2.78174603  1.30224782 12.3633157   0.21338772  1.2855309
 10.11544012]
{np.int64(0): np.float64(4.37305053025577), np.int64(1): np.float64(2.7817460317460316), np.int64(2): np.float64(1.3022478172023035), np.int64(3): np.float64(12.36331569664903), np.int64(4): np.float64(0.21338772031292808), np.int64(5): np.float64(1.285530900421786), np.int64(6): np.float64(10.115440115440116)}


In [31]:
for images, labels in train_ds.take(1):
    print(
        tf.reduce_min(images).numpy()
    )
    print(
        tf.reduce_max(images).numpy()
    )

0.0
255.0


In [32]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 82s 5us/step


In [33]:
efficientnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax")
])

In [34]:
efficientnet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [23]:
history_eff = efficientnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 76s 326ms/step - accuracy: 0.3876 - loss: 1.6701 - val_accuracy: 0.5133 - val_loss: 1.3917
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 322ms/step - accuracy: 0.4869 - loss: 1.3757 - val_accuracy: 0.5340 - val_loss: 1.2773
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 327ms/step - accuracy: 0.5245 - loss: 1.2242 - val_accuracy: 0.5839 - val_loss: 1.1261
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 72s 320ms/step - accuracy: 0.5285 - loss: 1.2242 - val_accuracy: 0.5499 - val_loss: 1.1657
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 72s 321ms/step - accuracy: 0.5367 - loss: 1.1498 - val_accuracy: 0.5885 - val_loss: 1.0459
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 73s 326ms/step - accuracy: 0.5459 - loss: 1.1259 - val_accuracy: 0.6298 - val_loss: 0.9728
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 328ms/step - accuracy: 0.5605 - loss: 1.1013 - val_accuracy: 0.5406 - val_loss: 1.0755
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 74s 328ms/step - accuracy: 0.5602 - loss: 1.0328 - val

In [24]:
test_loss, test_acc = efficientnet.evaluate(test_ds)
print("Accuracy:",test_acc)
print("Loss:",test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 13s 267ms/step - accuracy: 0.6420 - loss: 0.9446
Accuracy: 0.642049252986908
Loss: 0.9446136951446533


In [35]:
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
resnet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
resnet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 281s 3us/step


In [25]:
history_resnet = resnet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 21s 0us/step
Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 205s 915ms/step - accuracy: 0.4756 - loss: 1.5036 - val_accuracy: 0.5806 - val_loss: 1.1212
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 204s 918ms/step - accuracy: 0.5773 - loss: 1.1511 - val_accuracy: 0.6565 - val_loss: 0.9121
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 200s 899ms/step - accuracy: 0.5680 - loss: 1.1431 - val_accuracy: 0.5959 - val_loss: 1.0775
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 203s 915ms/step - accuracy: 0.6130 - loss: 0.9646 - val_accuracy: 0.6511 - val_loss: 0.9338
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 201s 903ms/step - accuracy: 0.6243 - loss: 0.9117 - val_accuracy: 0.5806 - val_loss: 1.1022
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 199s 898ms/step - accuracy: 0.6475 - loss: 0.8541 - val_accuracy: 0.6858 - val_loss: 0.8274
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 201s 907ms/step - accuracy: 0.6665 - loss: 0.8145 - val_accuracy: 0.6951 - val_loss: 0.8603
Epoch 8/10
220/220 ━━━━━

In [26]:
test_loss, test_acc = resnet.evaluate(test_ds)
print("Accuracy:",test_acc)
print("Loss:",test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 36s 756ms/step - accuracy: 0.6820 - loss: 0.8218
Accuracy: 0.681969404220581
Loss: 0.8218181729316711


In [36]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)
base_model.trainable = False
densenet = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
])
densenet.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 59s 2us/step


In [27]:
history_dense = densenet.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 197s 872ms/step - accuracy: 0.3414 - loss: 2.5769 - val_accuracy: 0.4341 - val_loss: 1.5020
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 197s 888ms/step - accuracy: 0.3990 - loss: 1.7304 - val_accuracy: 0.3142 - val_loss: 1.8504
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 198s 893ms/step - accuracy: 0.4258 - loss: 1.5854 - val_accuracy: 0.4567 - val_loss: 1.4148
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 195s 879ms/step - accuracy: 0.4418 - loss: 1.4799 - val_accuracy: 0.5213 - val_loss: 1.1375
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 196s 884ms/step - accuracy: 0.4452 - loss: 1.4678 - val_accuracy: 0.4927 - val_loss: 1.2538
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 195s 880ms/step - accuracy: 0.4833 - loss: 1.3840 - val_accuracy: 0.4454 - val_loss: 1.3880
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 193s 870ms/step - accuracy: 0.4796 - loss: 1.3662 - val_accuracy: 0.3755 - val_loss: 1.5503
Epoch 8/10
220/220 ━━━━━━

In [28]:
test_loss, test_acc = densenet.evaluate(test_ds)
print("Accuracy:",test_acc)
print("Loss:",test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 34s 716ms/step - accuracy: 0.5655 - loss: 0.9847
Accuracy: 0.5655356049537659
Loss: 0.9846996068954468


In [37]:
basic_cnn.save("basic_cnn.keras")
deep_cnn.save("deep_cnn.keras")
cnn_bn.save("cnn_batchnorm.keras")
cnn_dropout.save("cnn_dropout.keras")
mobilenet.save("mobilenet.keras")
efficientnet.save("efficientnet.keras")
resnet.save("resnet.keras")
densenet.save("densenet.keras")